In [4]:
import os
import json
import zipfile
import pandas as pd
import numpy as np

BASE_DIR = r"."   
FLIGHT_ZIP = os.path.join(BASE_DIR, "flightdata.zip")

AIRPORTS_CSV = os.path.join(BASE_DIR, "CIS4400_project08_references_airports.csv")
AIRLINES_CSV = os.path.join(BASE_DIR, "CIS4400_project08_references_airlines.csv")
AIRCRAFT_TYPES_CSV = os.path.join(BASE_DIR, "CIS4400_project08_references_aircrafts_types.csv")
AIRPLANES_CSV = os.path.join(BASE_DIR, "CIS4400_project08_references_airplanes.csv")
CITIES_CSV = os.path.join(BASE_DIR, "CIS4400_project08_references_cities.csv")
COUNTRIES_CSV = os.path.join(BASE_DIR, "CIS4400_project08_references_countries.csv")

OUTPUT_DIR = os.path.join(BASE_DIR, "warehouse_output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_JSON_FILES = None  

def clean_string(series):
    s = series.astype("string").str.strip()
    return s.replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "NONE": pd.NA,
        "NaN": pd.NA
    })

def safe_str(series):
    return clean_string(series).str.upper()

def normalize_code(series):
    return clean_string(series).str.upper()

def to_datetime_safe(series):
    return pd.to_datetime(series, errors="coerce", utc=True)

def bool_to_int(series):
    return series.fillna(False).astype(int)

def make_surrogate_key(df, key_name, start=1):
    df = df.reset_index(drop=True).copy()
    df[key_name] = range(start, start + len(df))
    cols = [key_name] + [c for c in df.columns if c != key_name]
    return df[cols]

def read_json_files_from_zip(zip_path, max_files=None):
    rows = []
    with zipfile.ZipFile(zip_path, "r") as zf:
        json_files = [n for n in zf.namelist() if n.endswith(".json")]
        json_files = sorted(json_files)

        if max_files is not None:
            json_files = json_files[:max_files]

        print(f"Reading {len(json_files)} JSON files...")

        for i, file_name in enumerate(json_files, 1):
            try:
                raw = zf.read(file_name)
                data = json.loads(raw)

                if isinstance(data, list):
                    for rec in data:
                        rec["source_file_name"] = os.path.basename(file_name)
                        rows.append(rec)
                else:
                    print(f"Skipping unexpected JSON structure in {file_name}")

                if i % 100 == 0:
                    print(f"Processed {i} files...")

            except Exception as e:
                print(f"Error reading {file_name}: {e}")

    return pd.DataFrame(rows)

def get_nested_value(obj, key):
    if isinstance(obj, dict):
        return obj.get(key)
    return None

flights_raw = read_json_files_from_zip(FLIGHT_ZIP, MAX_JSON_FILES)

print("Raw flight rows:", len(flights_raw))
print("Columns:", flights_raw.columns.tolist())

flights = pd.DataFrame()

flights["source_file_name"] = flights_raw["source_file_name"]
flights["flight_date"] = flights_raw["flight_date"]
flights["flight_status"] = flights_raw["flight_status"]

# Departure
flights["departure_airport"] = flights_raw["departure"].apply(lambda x: get_nested_value(x, "airport"))
flights["departure_timezone"] = flights_raw["departure"].apply(lambda x: get_nested_value(x, "timezone"))
flights["departure_iata"] = flights_raw["departure"].apply(lambda x: get_nested_value(x, "iata"))
flights["departure_icao"] = flights_raw["departure"].apply(lambda x: get_nested_value(x, "icao"))
flights["departure_terminal"] = flights_raw["departure"].apply(lambda x: get_nested_value(x, "terminal"))
flights["departure_gate"] = flights_raw["departure"].apply(lambda x: get_nested_value(x, "gate"))
flights["departure_delay_minutes"] = flights_raw["departure"].apply(lambda x: get_nested_value(x, "delay"))
flights["departure_scheduled_ts"] = flights_raw["departure"].apply(lambda x: get_nested_value(x, "scheduled"))
flights["departure_estimated_ts"] = flights_raw["departure"].apply(lambda x: get_nested_value(x, "estimated"))
flights["departure_actual_ts"] = flights_raw["departure"].apply(lambda x: get_nested_value(x, "actual"))
flights["departure_estimated_runway_ts"] = flights_raw["departure"].apply(lambda x: get_nested_value(x, "estimated_runway"))
flights["departure_actual_runway_ts"] = flights_raw["departure"].apply(lambda x: get_nested_value(x, "actual_runway"))

# Arrival
flights["arrival_airport"] = flights_raw["arrival"].apply(lambda x: get_nested_value(x, "airport"))
flights["arrival_timezone"] = flights_raw["arrival"].apply(lambda x: get_nested_value(x, "timezone"))
flights["arrival_iata"] = flights_raw["arrival"].apply(lambda x: get_nested_value(x, "iata"))
flights["arrival_icao"] = flights_raw["arrival"].apply(lambda x: get_nested_value(x, "icao"))
flights["arrival_terminal"] = flights_raw["arrival"].apply(lambda x: get_nested_value(x, "terminal"))
flights["arrival_gate"] = flights_raw["arrival"].apply(lambda x: get_nested_value(x, "gate"))
flights["arrival_baggage"] = flights_raw["arrival"].apply(lambda x: get_nested_value(x, "baggage"))
flights["arrival_delay_minutes"] = flights_raw["arrival"].apply(lambda x: get_nested_value(x, "delay"))
flights["arrival_scheduled_ts"] = flights_raw["arrival"].apply(lambda x: get_nested_value(x, "scheduled"))
flights["arrival_estimated_ts"] = flights_raw["arrival"].apply(lambda x: get_nested_value(x, "estimated"))
flights["arrival_actual_ts"] = flights_raw["arrival"].apply(lambda x: get_nested_value(x, "actual"))
flights["arrival_estimated_runway_ts"] = flights_raw["arrival"].apply(lambda x: get_nested_value(x, "estimated_runway"))
flights["arrival_actual_runway_ts"] = flights_raw["arrival"].apply(lambda x: get_nested_value(x, "actual_runway"))

# Airline
flights["airline_name"] = flights_raw["airline"].apply(lambda x: get_nested_value(x, "name"))
flights["airline_iata"] = flights_raw["airline"].apply(lambda x: get_nested_value(x, "iata"))
flights["airline_icao"] = flights_raw["airline"].apply(lambda x: get_nested_value(x, "icao"))

# Flight
flights["flight_number"] = flights_raw["flight"].apply(lambda x: get_nested_value(x, "number"))
flights["flight_iata"] = flights_raw["flight"].apply(lambda x: get_nested_value(x, "iata"))
flights["flight_icao"] = flights_raw["flight"].apply(lambda x: get_nested_value(x, "icao"))
flights["codeshared_raw"] = flights_raw["flight"].apply(lambda x: get_nested_value(x, "codeshared"))

# Aircraft
flights["aircraft_registration"] = flights_raw["aircraft"].apply(lambda x: get_nested_value(x, "registration"))
flights["aircraft_iata"] = flights_raw["aircraft"].apply(lambda x: get_nested_value(x, "iata"))
flights["aircraft_icao24"] = flights_raw["aircraft"].apply(lambda x: get_nested_value(x, "icao24"))
flights["aircraft_icao"] = flights_raw["aircraft"].apply(lambda x: get_nested_value(x, "icao"))

# Live
flights["live_updated"] = flights_raw["live"].apply(lambda x: get_nested_value(x, "updated"))
flights["live_latitude"] = flights_raw["live"].apply(lambda x: get_nested_value(x, "latitude"))
flights["live_longitude"] = flights_raw["live"].apply(lambda x: get_nested_value(x, "longitude"))
flights["live_altitude"] = flights_raw["live"].apply(lambda x: get_nested_value(x, "altitude"))
flights["live_direction"] = flights_raw["live"].apply(lambda x: get_nested_value(x, "direction"))
flights["live_speed_horizontal"] = flights_raw["live"].apply(lambda x: get_nested_value(x, "speed_horizontal"))
flights["live_speed_vertical"] = flights_raw["live"].apply(lambda x: get_nested_value(x, "speed_vertical"))
flights["live_is_ground"] = flights_raw["live"].apply(lambda x: get_nested_value(x, "is_ground"))

print("Flattened rows:", len(flights))

# normalizing key text fields
code_cols = [
    "departure_iata", "departure_icao", "arrival_iata", "arrival_icao",
    "airline_iata", "airline_icao", "flight_iata", "flight_icao",
    "aircraft_iata", "aircraft_icao", "aircraft_icao24"
]
for col in code_cols:
    flights[col] = normalize_code(flights[col])

text_cols = [
    "departure_airport", "departure_timezone", "departure_terminal", "departure_gate",
    "arrival_airport", "arrival_timezone", "arrival_terminal", "arrival_gate",
    "arrival_baggage", "airline_name", "flight_number", "aircraft_registration"
]
for col in text_cols:
    flights[col] = clean_string(flights[col])

datetime_cols = [
    "departure_scheduled_ts", "departure_estimated_ts", "departure_actual_ts",
    "departure_estimated_runway_ts", "departure_actual_runway_ts",
    "arrival_scheduled_ts", "arrival_estimated_ts", "arrival_actual_ts",
    "arrival_estimated_runway_ts", "arrival_actual_runway_ts",
    "live_updated"
]
for col in datetime_cols:
    flights[col] = to_datetime_safe(flights[col])

flights["flight_date"] = pd.to_datetime(flights["flight_date"], errors="coerce").dt.date

numeric_cols = [
    "departure_delay_minutes", "arrival_delay_minutes",
    "live_latitude", "live_longitude", "live_altitude",
    "live_direction", "live_speed_horizontal", "live_speed_vertical"
]
for col in numeric_cols:
    flights[col] = pd.to_numeric(flights[col], errors="coerce")

flights["departure_delay_minutes"] = flights["departure_delay_minutes"].fillna(0)
flights["arrival_delay_minutes"] = flights["arrival_delay_minutes"].fillna(0)

# flags from status
status_clean = clean_string(flights["flight_status"]).str.lower()
flights["flight_status"] = status_clean.fillna("unknown")

flights["is_cancelled"] = (flights["flight_status"] == "cancelled").astype(int)
flights["is_delayed"] = (flights["flight_status"] == "delayed").astype(int)
flights["is_landed"] = (flights["flight_status"] == "landed").astype(int)
flights["is_active"] = (flights["flight_status"] == "active").astype(int)
flights["is_scheduled"] = (flights["flight_status"] == "scheduled").astype(int)
flights["is_diverted"] = (flights["flight_status"] == "diverted").astype(int)

flights["is_live_tracked"] = flights["live_updated"].notna().astype(int)
flights["is_codeshare"] = flights["codeshared_raw"].notna().astype(int)
flights["flight_count"] = 1

airports_ref = pd.read_csv(AIRPORTS_CSV)
airlines_ref = pd.read_csv(AIRLINES_CSV)
aircraft_types_ref = pd.read_csv(AIRCRAFT_TYPES_CSV)
airplanes_ref = pd.read_csv(AIRPLANES_CSV)
cities_ref = pd.read_csv(CITIES_CSV)
countries_ref = pd.read_csv(COUNTRIES_CSV)

# clean references
airports_ref["iata_code"] = normalize_code(airports_ref["iata_code"])
airports_ref["icao_code"] = normalize_code(airports_ref["icao_code"])
airports_ref["airport_name"] = clean_string(airports_ref["airport_name"])
airports_ref["country_iso2"] = normalize_code(airports_ref["country_iso2"])
airports_ref["country_name"] = clean_string(airports_ref["country_name"])
airports_ref["city_iata_code"] = normalize_code(airports_ref["city_iata_code"])
airports_ref["timezone"] = clean_string(airports_ref["timezone"])
airports_ref["gmt"] = clean_string(airports_ref["gmt"])
airports_ref["latitude"] = pd.to_numeric(airports_ref["latitude"], errors="coerce")
airports_ref["longitude"] = pd.to_numeric(airports_ref["longitude"], errors="coerce")
airports_ref = airports_ref.drop_duplicates(subset=["iata_code"], keep="first").copy()

airlines_ref["iata_code"] = normalize_code(airlines_ref["iata_code"])
airlines_ref["icao_code"] = normalize_code(airlines_ref["icao_code"])
airlines_ref["airline_name"] = clean_string(airlines_ref["airline_name"])
airlines_ref["country_iso2"] = normalize_code(airlines_ref["country_iso2"])
airlines_ref["country_name"] = clean_string(airlines_ref["country_name"])
airlines_ref["callsign"] = clean_string(airlines_ref["callsign"])
airlines_ref["hub_code"] = normalize_code(airlines_ref["hub_code"])
airlines_ref = airlines_ref[airlines_ref["iata_code"].notna()].copy()
airlines_ref = airlines_ref.drop_duplicates(subset=["iata_code"], keep="first").copy()

aircraft_types_ref["iata_code"] = normalize_code(aircraft_types_ref["iata_code"])
aircraft_types_ref["aircraft_name"] = clean_string(aircraft_types_ref["aircraft_name"])
aircraft_types_ref = aircraft_types_ref.drop_duplicates(subset=["iata_code"], keep="first").copy()

airplanes_ref["iata_type"] = normalize_code(airplanes_ref["iata_type"])
airplanes_ref["iata_code_long"] = normalize_code(airplanes_ref["iata_code_long"])
airplanes_ref["iata_code_short"] = normalize_code(airplanes_ref["iata_code_short"])
airplanes_ref["model_code"] = clean_string(airplanes_ref["model_code"])
airplanes_ref["model_name"] = clean_string(airplanes_ref["model_name"])
airplanes_ref["plane_series"] = clean_string(airplanes_ref["plane_series"])
airplanes_ref["plane_class"] = clean_string(airplanes_ref["plane_class"])
airplanes_ref["production_line"] = clean_string(airplanes_ref["production_line"])
airplanes_ref["engines_type"] = clean_string(airplanes_ref["engines_type"])
airplanes_ref["registration_number"] = normalize_code(airplanes_ref["registration_number"])
airplanes_ref = airplanes_ref.drop_duplicates(subset=["registration_number"], keep="first").copy()

cities_ref["iata_code"] = normalize_code(cities_ref["iata_code"])
cities_ref["city_name"] = clean_string(cities_ref["city_name"])
cities_ref["country_iso2"] = normalize_code(cities_ref["country_iso2"])
cities_ref["timezone"] = clean_string(cities_ref["timezone"])
cities_ref = cities_ref.drop_duplicates(subset=["iata_code"], keep="first").copy()

countries_ref["country_iso2"] = normalize_code(countries_ref["country_iso2"])
countries_ref["country_name"] = clean_string(countries_ref["country_name"])
countries_ref["continent"] = clean_string(countries_ref["continent"])
countries_ref = countries_ref.drop_duplicates(subset=["country_iso2"], keep="first").copy()

# DIM DATE
dim_date = pd.DataFrame({
    "full_date": pd.to_datetime(flights["flight_date"], errors="coerce")
}).dropna().drop_duplicates()

dim_date["date_key"] = dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
dim_date["day_of_month"] = dim_date["full_date"].dt.day
dim_date["day_of_week"] = dim_date["full_date"].dt.dayofweek + 1
dim_date["day_name"] = dim_date["full_date"].dt.day_name()
dim_date["week_of_year"] = dim_date["full_date"].dt.isocalendar().week.astype(int)
dim_date["month_number"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["quarter_number"] = dim_date["full_date"].dt.quarter
dim_date["year_number"] = dim_date["full_date"].dt.year
dim_date["is_weekend"] = dim_date["day_of_week"].isin([6, 7]).astype(int)
dim_date["full_date"] = dim_date["full_date"].dt.date

dim_date = dim_date[
    ["date_key", "full_date", "day_of_month", "day_of_week", "day_name",
     "week_of_year", "month_number", "month_name", "quarter_number",
     "year_number", "is_weekend"]
].sort_values("date_key")

# DIM FLIGHT STATUS
dim_flight_status = flights[["flight_status"]].dropna().drop_duplicates().copy()
dim_flight_status["is_cancelled_flag"] = (dim_flight_status["flight_status"] == "cancelled").astype(int)
dim_flight_status["is_delayed_flag"] = (dim_flight_status["flight_status"] == "delayed").astype(int)
dim_flight_status["is_landed_flag"] = (dim_flight_status["flight_status"] == "landed").astype(int)
dim_flight_status["is_active_flag"] = (dim_flight_status["flight_status"] == "active").astype(int)
dim_flight_status["is_scheduled_flag"] = (dim_flight_status["flight_status"] == "scheduled").astype(int)
dim_flight_status["is_diverted_flag"] = (dim_flight_status["flight_status"] == "diverted").astype(int)
dim_flight_status = make_surrogate_key(dim_flight_status, "status_key")


# DIM AIRPORT
used_airports = pd.concat([
    flights[["departure_iata", "departure_icao", "departure_airport", "departure_timezone"]]
    .rename(columns={
        "departure_iata": "iata_code",
        "departure_icao": "icao_code",
        "departure_airport": "airport_name_json",
        "departure_timezone": "timezone_json"
    }),
    flights[["arrival_iata", "arrival_icao", "arrival_airport", "arrival_timezone"]]
    .rename(columns={
        "arrival_iata": "iata_code",
        "arrival_icao": "icao_code",
        "arrival_airport": "airport_name_json",
        "arrival_timezone": "timezone_json"
    })
], ignore_index=True)

used_airports["iata_code"] = normalize_code(used_airports["iata_code"])
used_airports["icao_code"] = normalize_code(used_airports["icao_code"])
used_airports["airport_name_json"] = clean_string(used_airports["airport_name_json"])
used_airports["timezone_json"] = clean_string(used_airports["timezone_json"])

used_airports = used_airports[
    used_airports["iata_code"].notna() | used_airports["icao_code"].notna()
].drop_duplicates().copy()

dim_airport = used_airports.merge(
    airports_ref[
        ["airport_id", "iata_code", "icao_code", "city_iata_code", "country_iso2",
         "geoname_id", "latitude", "longitude", "airport_name", "country_name",
         "phone_number", "timezone", "gmt"]
    ],
    on="iata_code",
    how="left",
    suffixes=("", "_ref"),
    validate="m:1"
)

dim_airport["airport_name"] = dim_airport["airport_name"].fillna(dim_airport["airport_name_json"])
dim_airport["timezone"] = dim_airport["timezone"].fillna(dim_airport["timezone_json"])
dim_airport["icao_code"] = dim_airport["icao_code_ref"].fillna(dim_airport["icao_code"])
dim_airport = dim_airport.drop(columns=["icao_code_ref"], errors="ignore")

dim_airport = dim_airport.merge(
    cities_ref[["iata_code", "city_name"]],
    left_on="city_iata_code",
    right_on="iata_code",
    how="left",
    suffixes=("", "_city"),
    validate="m:1"
).drop(columns=["iata_code_city"], errors="ignore")

dim_airport = dim_airport.merge(
    countries_ref[["country_iso2", "continent"]],
    on="country_iso2",
    how="left",
    validate="m:1"
)

dim_airport = dim_airport[
    ["airport_id", "iata_code", "icao_code", "airport_name", "city_iata_code",
     "city_name", "country_iso2", "country_name", "continent", "timezone",
     "gmt", "latitude", "longitude", "geoname_id", "phone_number"]
].copy()

dim_airport = dim_airport[
    dim_airport["iata_code"].notna() | dim_airport["icao_code"].notna()
].copy()

dim_airport = dim_airport.drop_duplicates(subset=["iata_code"], keep="first").copy()
dim_airport = make_surrogate_key(dim_airport, "airport_key")

# DIM AIRLINE
used_airlines = flights[["airline_iata", "airline_icao", "airline_name"]].drop_duplicates().copy()
used_airlines["airline_iata"] = normalize_code(used_airlines["airline_iata"])
used_airlines["airline_icao"] = normalize_code(used_airlines["airline_icao"])
used_airlines["airline_name"] = clean_string(used_airlines["airline_name"])

dim_airline = used_airlines.merge(
    airlines_ref[
        ["airline_id", "iata_code", "icao_code", "airline_name", "callsign",
         "hub_code", "country_iso2", "country_name", "date_founded",
         "fleet_size", "fleet_average_age", "iata_prefix_accounting",
         "status", "type"]
    ],
    left_on="airline_iata",
    right_on="iata_code",
    how="left",
    suffixes=("_json", "_ref"),
    validate="m:1"
)

dim_airline["airline_name"] = dim_airline["airline_name_ref"].fillna(dim_airline["airline_name_json"])

dim_airline = dim_airline.rename(columns={
    "airline_iata": "iata_code_json",
    "airline_icao": "icao_code_json",
    "status": "airline_status",
    "type": "airline_type"
})

dim_airline["iata_code"] = dim_airline["iata_code"].fillna(dim_airline["iata_code_json"])
dim_airline["icao_code"] = dim_airline["icao_code"].fillna(dim_airline["icao_code_json"])

dim_airline = dim_airline[
    ["airline_id", "iata_code", "icao_code", "airline_name", "callsign",
     "hub_code", "country_iso2", "country_name", "date_founded",
     "fleet_size", "fleet_average_age", "iata_prefix_accounting",
     "airline_status", "airline_type"]
].copy()

dim_airline = dim_airline[dim_airline["iata_code"].notna()].copy()
dim_airline = dim_airline.sort_values(["iata_code", "airline_name"], na_position="last")
dim_airline = dim_airline.drop_duplicates(subset=["iata_code"], keep="first").copy()
dim_airline = make_surrogate_key(dim_airline, "airline_key")

# DIM AIRCRAFT
used_aircraft = flights[["aircraft_iata", "aircraft_registration"]].drop_duplicates().copy()
used_aircraft["aircraft_iata"] = normalize_code(used_aircraft["aircraft_iata"])
used_aircraft["aircraft_registration"] = normalize_code(used_aircraft["aircraft_registration"])

dim_aircraft = used_aircraft.merge(
    aircraft_types_ref[["id", "iata_code", "aircraft_name", "plane_type_id"]],
    left_on="aircraft_iata",
    right_on="iata_code",
    how="left",
    validate="m:1"
)

dim_aircraft = dim_aircraft.merge(
    airplanes_ref[
        ["registration_number", "iata_type", "iata_code_long", "iata_code_short",
         "model_code", "model_name", "plane_series", "plane_class",
         "production_line", "engines_count", "engines_type", "plane_status"]
    ],
    left_on="aircraft_registration",
    right_on="registration_number",
    how="left",
    validate="m:1"
)

dim_aircraft["aircraft_name"] = dim_aircraft["aircraft_name"].fillna(dim_aircraft["model_name"])

dim_aircraft = dim_aircraft.rename(columns={
    "id": "aircraft_type_ref_id",
    "aircraft_iata": "iata_type_code"
})

dim_aircraft = dim_aircraft[
    ["iata_type_code", "aircraft_registration", "aircraft_type_ref_id",
     "plane_type_id", "aircraft_name", "iata_code_long", "iata_code_short",
     "model_code", "model_name", "plane_series", "plane_class",
     "production_line", "engines_count", "engines_type", "plane_status"]
].copy()

dim_aircraft = dim_aircraft.drop_duplicates(
    subset=["iata_type_code", "aircraft_registration"],
    keep="first"
).copy()

dim_aircraft = make_surrogate_key(dim_aircraft, "aircraft_key")

# DIM ROUTE
airport_lookup_for_route = dim_airport[["airport_key", "iata_code", "country_name", "timezone"]].drop_duplicates("iata_code").copy()

route_base = flights[
    ["departure_iata", "departure_icao", "arrival_iata", "arrival_icao"]
].drop_duplicates().copy()

route_base = route_base.merge(
    airport_lookup_for_route,
    left_on="departure_iata",
    right_on="iata_code",
    how="left",
    validate="m:1"
).rename(columns={
    "country_name": "origin_country_name",
    "timezone": "origin_timezone",
    "airport_key": "origin_airport_lookup_key"
}).drop(columns=["iata_code"])

route_base = route_base.merge(
    airport_lookup_for_route,
    left_on="arrival_iata",
    right_on="iata_code",
    how="left",
    validate="m:1"
).rename(columns={
    "country_name": "destination_country_name",
    "timezone": "destination_timezone",
    "airport_key": "destination_airport_lookup_key"
}).drop(columns=["iata_code"])

route_base["route_name"] = route_base["departure_iata"].fillna("UNK") + "-" + route_base["arrival_iata"].fillna("UNK")
route_base["is_domestic"] = (
    route_base["origin_country_name"].notna() &
    route_base["destination_country_name"].notna() &
    (route_base["origin_country_name"] == route_base["destination_country_name"])
).astype(int)
route_base["is_international"] = (
    route_base["origin_country_name"].notna() &
    route_base["destination_country_name"].notna() &
    (route_base["origin_country_name"] != route_base["destination_country_name"])
).astype(int)

dim_route = route_base[
    ["departure_iata", "departure_icao", "arrival_iata", "arrival_icao",
     "route_name", "is_domestic", "is_international",
     "origin_country_name", "destination_country_name",
     "origin_timezone", "destination_timezone"]
].drop_duplicates().copy()

dim_route = dim_route.rename(columns={
    "departure_iata": "origin_iata_code",
    "departure_icao": "origin_icao_code",
    "arrival_iata": "destination_iata_code",
    "arrival_icao": "destination_icao_code"
})

dim_route = dim_route.drop_duplicates(
    subset=["origin_iata_code", "destination_iata_code"],
    keep="first"
).copy()

dim_route = make_surrogate_key(dim_route, "route_key")

fact_flight = flights.copy()

# date_key
fact_flight["date_key"] = pd.to_datetime(fact_flight["flight_date"], errors="coerce").dt.strftime("%Y%m%d")
fact_flight["date_key"] = pd.to_numeric(fact_flight["date_key"], errors="coerce").astype("Int64")

#  status_key
fact_flight = fact_flight.merge(
    dim_flight_status[["status_key", "flight_status"]],
    on="flight_status",
    how="left",
    validate="m:1"
)

# airport lookups
departure_airport_lookup = dim_airport[["airport_key", "iata_code"]].drop_duplicates("iata_code").rename(columns={
    "airport_key": "departure_airport_key",
    "iata_code": "departure_iata_join"
})

arrival_airport_lookup = dim_airport[["airport_key", "iata_code"]].drop_duplicates("iata_code").rename(columns={
    "airport_key": "arrival_airport_key",
    "iata_code": "arrival_iata_join"
})

# departure airport key
fact_flight = fact_flight.merge(
    departure_airport_lookup,
    left_on="departure_iata",
    right_on="departure_iata_join",
    how="left",
    validate="m:1"
).drop(columns=["departure_iata_join"])

# arrival airport key
fact_flight = fact_flight.merge(
    arrival_airport_lookup,
    left_on="arrival_iata",
    right_on="arrival_iata_join",
    how="left",
    validate="m:1"
).drop(columns=["arrival_iata_join"])

# airline lookup
airline_lookup = dim_airline[["airline_key", "iata_code"]].drop_duplicates("iata_code").rename(columns={
    "iata_code": "airline_iata_join"
})

fact_flight = fact_flight.merge(
    airline_lookup,
    left_on="airline_iata",
    right_on="airline_iata_join",
    how="left",
    validate="m:1"
).drop(columns=["airline_iata_join"])

# aircraft lookup
aircraft_lookup = dim_aircraft[["aircraft_key", "iata_type_code", "aircraft_registration"]].drop_duplicates(
    subset=["iata_type_code", "aircraft_registration"]
).rename(columns={
    "iata_type_code": "aircraft_iata_join",
    "aircraft_registration": "aircraft_registration_join"
})

fact_flight = fact_flight.merge(
    aircraft_lookup,
    left_on=["aircraft_iata", "aircraft_registration"],
    right_on=["aircraft_iata_join", "aircraft_registration_join"],
    how="left",
    validate="m:1"
).drop(columns=["aircraft_iata_join", "aircraft_registration_join"])

# route lookup
route_lookup = dim_route[["route_key", "origin_iata_code", "destination_iata_code"]].drop_duplicates(
    subset=["origin_iata_code", "destination_iata_code"]
)

fact_flight = fact_flight.merge(
    route_lookup,
    left_on=["departure_iata", "arrival_iata"],
    right_on=["origin_iata_code", "destination_iata_code"],
    how="left",
    validate="m:1"
).drop(columns=["origin_iata_code", "destination_iata_code"])

# make final fact primary keys
fact_flight = make_surrogate_key(fact_flight, "flight_fact_key")

# final fact columns
fact_flight_final = fact_flight[
    [
        "flight_fact_key",
        "date_key",
        "departure_airport_key",
        "arrival_airport_key",
        "airline_key",
        "status_key",
        "aircraft_key",
        "route_key",
        "source_file_name",
        "flight_date",
        "flight_number",
        "flight_iata",
        "flight_icao",
        "departure_terminal",
        "departure_gate",
        "arrival_terminal",
        "arrival_gate",
        "arrival_baggage",
        "departure_scheduled_ts",
        "departure_estimated_ts",
        "departure_actual_ts",
        "departure_estimated_runway_ts",
        "departure_actual_runway_ts",
        "arrival_scheduled_ts",
        "arrival_estimated_ts",
        "arrival_actual_ts",
        "arrival_estimated_runway_ts",
        "arrival_actual_runway_ts",
        "departure_delay_minutes",
        "arrival_delay_minutes",
        "is_codeshare",
        "is_live_tracked",
        "is_cancelled",
        "is_delayed",
        "is_landed",
        "is_active",
        "is_scheduled",
        "is_diverted",
        "flight_count",
        "live_latitude",
        "live_longitude",
        "live_altitude",
        "live_direction",
        "live_speed_horizontal",
        "live_speed_vertical",
        "live_is_ground"
    ]
]

dim_date.to_csv(os.path.join(OUTPUT_DIR, "dim_date.csv"), index=False)
dim_airport.to_csv(os.path.join(OUTPUT_DIR, "dim_airport.csv"), index=False)
dim_airline.to_csv(os.path.join(OUTPUT_DIR, "dim_airline.csv"), index=False)
dim_flight_status.to_csv(os.path.join(OUTPUT_DIR, "dim_flight_status.csv"), index=False)
dim_aircraft.to_csv(os.path.join(OUTPUT_DIR, "dim_aircraft.csv"), index=False)
dim_route.to_csv(os.path.join(OUTPUT_DIR, "dim_route.csv"), index=False)
fact_flight_final.to_csv(os.path.join(OUTPUT_DIR, "fact_flight.csv"), index=False)

print("\nDone. Output files saved to:")
print(OUTPUT_DIR)

print("\nRow counts:")
print("dim_date:", len(dim_date))
print("dim_airport:", len(dim_airport))
print("dim_airline:", len(dim_airline))
print("dim_flight_status:", len(dim_flight_status))
print("dim_aircraft:", len(dim_aircraft))
print("dim_route:", len(dim_route))
print("fact_flight:", len(fact_flight_final))

Reading 1156 JSON files...
Processed 100 files...
Processed 200 files...
Processed 300 files...
Processed 400 files...
Processed 500 files...
Processed 600 files...
Processed 700 files...
Processed 800 files...
Processed 900 files...
Processed 1000 files...
Processed 1100 files...
Raw flight rows: 995311
Columns: ['flight_date', 'flight_status', 'departure', 'arrival', 'airline', 'flight', 'aircraft', 'live', 'source_file_name']
Flattened rows: 995311

Done. Output files saved to:
.\warehouse_output

Row counts:
dim_date: 398
dim_airport: 762
dim_airline: 210
dim_flight_status: 6
dim_aircraft: 7729
dim_route: 1385
fact_flight: 995311


In [5]:
print(fact_flight.shape)
print(dim_airport.shape)
print(dim_airline.shape)
print(dim_route.shape)

(995311, 64)
(762, 16)
(210, 15)
(1385, 12)


In [6]:
print(dim_date.head())
print(dim_airport.head())
print(dim_airline.head())
print(dim_flight_status.head())
print(dim_aircraft.head())
print(dim_route.head())
print(fact_flight_final.head())

print(fact_flight_final[[
    "date_key","departure_airport_key","arrival_airport_key",
    "airline_key","status_key","aircraft_key","route_key"
]].isna().sum())

       date_key   full_date  day_of_month  day_of_week   day_name  \
0      20250101  2025-01-01             1            3  Wednesday   
2819   20250102  2025-01-02             2            4   Thursday   
5978   20250103  2025-01-03             3            5     Friday   
9083   20250104  2025-01-04             4            6   Saturday   
12097  20250105  2025-01-05             5            7     Sunday   

       week_of_year  month_number month_name  quarter_number  year_number  \
0                 1             1    January               1         2025   
2819              1             1    January               1         2025   
5978              1             1    January               1         2025   
9083              1             1    January               1         2025   
12097             1             1    January               1         2025   

       is_weekend  
0               0  
2819            0  
5978            0  
9083            1  
12097           1  
  

In [7]:
print("dim_date:", dim_date.shape)
print("dim_airport:", dim_airport.shape)
print("dim_airline:", dim_airline.shape)
print("dim_flight_status:", dim_flight_status.shape)
print("dim_aircraft:", dim_aircraft.shape)
print("dim_route:", dim_route.shape)
print("fact_flight:", fact_flight_final.shape)

dim_date: (398, 11)
dim_airport: (762, 16)
dim_airline: (210, 15)
dim_flight_status: (6, 8)
dim_aircraft: (7729, 16)
dim_route: (1385, 12)
fact_flight: (995311, 46)
